# Demo: Default Config + Default Topology JSON

Este notebook tem o objetivo de demonstrar o uso dos parâmtros de simulação pré-configurados nos arquivos .yaml e .json para parâmetros da rede e da topologia, respectivamente, presentes em quantumnet/config.
- `quantumnet/config/default_config.yaml`
- `quantumnet/config/default_topology.json`

In [1]:
import sys
from pathlib import Path

if str(Path("..").resolve()) not in sys.path:
    sys.path.append(str(Path("..").resolve()))

from quantumnet.config import SimulationConfig
from quantumnet.control import Controller
from quantumnet.runtime import Clock
from quantumnet.topology import Network

In [2]:
repo_root = Path("..").resolve()
config_dir = repo_root / "quantumnet" / "config"

default_config_path = config_dir / "default_config.yaml"
default_topology_path = config_dir / "default_topology.json"

print("Config:", default_config_path)
print("Topology JSON:", default_topology_path)
print("Config exists?", default_config_path.exists())
print("Topology exists?", default_topology_path.exists())

Config: C:\Users\artue\OneDrive\Documentos\GitHub\QuantumNet\quantumnet\config\default_config.yaml
Topology JSON: C:\Users\artue\OneDrive\Documentos\GitHub\QuantumNet\quantumnet\config\default_topology.json
Config exists? True
Topology exists? True


Para utilizar o arquivo .json para a topologia, é necessário construí-lo ou editá-lo conforme suas preferências, adicionando e conectando nós. Esse arquivo guarda a topologia da rede, para utilizá-lo em sua simulação, é necessário alterar o parâmetro `config.topology.name` para Json e nomear em `config.topology.args` de acordo com o arquivo que se deseja carregar a topologia.

In [3]:
config = SimulationConfig.from_yaml(str(default_config_path))

# Usa somente o JSON de topologia default
config.topology.name = "Json"
config.topology.args = ["default_topology.json"]

network = Network(clock=Clock(), config=config)
network.set_ready_topology()

print("Topology name:", config.topology.name)
print("Topology arg:", config.topology.args)
print("Nodes:", list(network.nodes))
print("Edges:", list(network.edges))
print("Host map (name -> id):", network.host_name_to_id)

Topology name: Json
Topology arg: ['default_topology.json']
Nodes: [0, 1, 2, 3, 4]
Edges: [(0, 1), (1, 2), (2, 3), (3, 4)]
Host map (name -> id): {'0': 0, '1': 1, '2': 2, '3': 3, '4': 4}


In [4]:
# Preenche tabelas de roteamento de todos os hosts
controller = Controller(network)
controller.register_routing_tables()

print("Routing table por host:\n")
for host_id in sorted(network.hosts):
    host = network.hosts[host_id]
    print(f"Host {host_id} ({host.name})")
    print(host.routing_table)
    print()

Routing table por host:

Host 0 (0)
{0: [0], 1: [0, 1], 2: [0, 1, 2], 3: [0, 1, 2, 3], 4: [0, 1, 2, 3, 4]}

Host 1 (1)
{1: [1], 0: [1, 0], 2: [1, 2], 3: [1, 2, 3], 4: [1, 2, 3, 4]}

Host 2 (2)
{2: [2], 1: [2, 1], 3: [2, 3], 0: [2, 1, 0], 4: [2, 3, 4]}

Host 3 (3)
{3: [3], 2: [3, 2], 4: [3, 4], 1: [3, 2, 1], 0: [3, 2, 1, 0]}

Host 4 (4)
{4: [4], 3: [4, 3], 2: [4, 3, 2], 1: [4, 3, 2, 1], 0: [4, 3, 2, 1, 0]}

